# NeuroForecast v2: check every number, run the tool, see where it wins and where it fails

**Day-7 forecasting and trust triage for neuronal MEA assays.** AI4S Open Innovation: AI for Life Science, category **Tool & Platform**. Repository: <https://github.com/SaltyTaro/neuroforecast>.

At the day-7 recording of a 12-day network-formation assay, the tool says three things from day-5 and day-7 wells alone:
- whether the batch's reference is usable;
- what day 12 will show, always beside "carry day 7 forward";
- which conditions it cannot predict.

This notebook:
1. recomputes every published number from the per-case evidence;
2. runs the tool on a batch it never saw, and on a batch whose controls were silent;
3. recomputes both sealed tests, including the one the forecast lost.

Everything runs on a CPU in about a minute, from files in the repository. The optional last cell rebuilds the external test's development stage from EPA's files.

## 0. Setup

This cell clones the repository if needed and checks the package versions. The shipped models were saved with scikit-learn 1.9.0. If that version is missing, the cell installs the pinned requirements and stops, so the kernel can be restarted.

In [1]:
import subprocess, sys
from pathlib import Path

REPO = "https://github.com/SaltyTaro/neuroforecast.git"
candidates = [Path("."), Path(".."), Path("neuroforecast")]
root = next((p for p in candidates if (p / "tools/neuroforecast_triage_v2.py").exists()), None)
if root is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO, "neuroforecast"], check=True)
    root = Path("neuroforecast")
root = root.resolve()
sys.path.insert(0, str(root / "tools"))

import warnings
import sklearn, numpy, pandas
warnings.filterwarnings("ignore", category=pandas.errors.PerformanceWarning)  # raised inside a frozen core; harmless
if sklearn.__version__ != "1.9.0":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "tools/requirements-neuroforecast.txt")], check=True)
    raise SystemExit("Installed the pinned packages. Restart the kernel and run all cells again.")
print("repository:", root.name)
print("python", sys.version.split()[0], "| scikit-learn", sklearn.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)

repository: neuroforecast
python 3.14.2 | scikit-learn 1.9.0 | numpy 2.4.2 | pandas 3.0.0


## 1. Do the published numbers match the evidence?

Run this check first. `verify_published_numbers.py` recomputes every number quoted in the README, the report and the results documents from the per-case tables in `evaluation/`. It compares each one at the precision it is written to, and fails loudly if any differs.

In [2]:
result = subprocess.run([sys.executable, "-X", "utf8", "-B", "tools/verify_published_numbers.py"], capture_output=True, text=True, cwd=root)
lines = result.stdout.strip().splitlines()
print("\n".join(lines[:5] + ["..."] + lines[-3:]) if lines else result.stderr[-2000:])
assert result.returncode == 0, "a published number does not match the evidence"

ok   reserve product date-macro MAE                             published 0.7716     recomputed 0.77162
ok   reserve persistence date-macro MAE                         published 0.6088     recomputed 0.60878
ok   reserve comparator date-macro MAE                          published 0.8005     recomputed 0.8005
ok   reserve control-reference date-macro MAE                   published 0.915      recomputed 0.91502
ok   reserve dates product beats persistence                    published 1          recomputed 1
...
ok   browser replay reserve-batch conditions                    published 245        recomputed 245

All 209 published values match the per-case evidence tables.


## 2. The tool on a batch it never saw

`examples/v2/external_20160120_day5_day7.csv` holds real day-5 and day-7 wells from one batch of the sealed external test (2016-01-20). No v2 fit saw this batch or any of its chemicals.

The tool **rejects** day-9 and day-12 rows, so it cannot use information a researcher would not have at day 7.

In [3]:
import pandas as pd
import neuroforecast_gate as gate
import neuroforecast_triage_v2 as t2

bundle = t2.load_bundle()
wells = pd.read_csv(root / "examples/v2/external_20160120_day5_day7.csv")
print("input days:", sorted(map(int, wells.DIV.unique())), "| wells:", wells[["Plate.SN", "well"]].drop_duplicates().shape[0])
table, summary = t2.triage(wells, bundle)
batch = summary["batches"][0]
print(f"batch {batch['batch']}: {batch['conditions']} conditions, day-7 reference {batch['day7_reference_activity_network_spikes']:.1f} network spikes")
print("batch verdict:", batch["batch_verdict"])
print("declined:", batch["forecasts_declined"], "of", batch["conditions"])
columns = ["name", "dose", "day7_observed_log2", "forecast_log2", "forecast_low", "forecast_high", "persistence_log2", "verdict"]
table[columns].round(2).head(14)

input days: [5, 7] | wells: 144


batch 20160120: 42 conditions, day-7 reference 22.2 network spikes
batch verdict: day-7 measurement quality is usable
declined: 3 of 42


,name,dose,day7_observed_log2,forecast_log2,forecast_low,forecast_high,persistence_log2,verdict
0,Fosthiazate,0.1,0.42,-0.05,-0.67,0.57,0.42,forecast usable
1,Fosthiazate,0.3,0.22,-0.33,-0.99,0.34,0.22,forecast usable
2,Fosthiazate,1.0,0.35,-0.09,-0.71,0.54,0.35,forecast usable
3,Fosthiazate,3.0,0.31,-0.07,-0.66,0.53,0.31,forecast usable
4,Fosthiazate,10.0,0.14,-0.01,-0.57,0.55,0.14,forecast usable
5,Fosthiazate,30.0,0.20,0.30,-0.32,0.92,0.20,forecast usable
6,Fosthiazate,100.0,-0.08,0.09,-0.50,0.67,-0.08,forecast usable
7,Tebupirimfos,0.1,-0.17,0.02,-0.54,0.57,-0.17,forecast usable
8,Tebupirimfos,0.3,-0.12,-0.01,-0.51,0.49,-0.12,forecast usable
9,Tebupirimfos,1.0,-0.27,0.05,-0.60,0.70,-0.27,forecast usable


Units are log2 of the condition's network-spike count relative to the same plate's untreated controls: 0 is "like control" and −1 is half.

The day-12 outcomes for this batch were sealed until the test was scored. Here is how the forecast and carry-forward did on it:

In [4]:
scored = pd.read_csv(root / "evaluation/external/arm_b_cases.csv", dtype={"identity": str})
check = table.merge(scored[["case_id", "target"]], on="case_id")
forecast_mae = gate.macro_mae(check.assign(prediction=check.forecast_log2))
carry_mae = gate.macro_mae(check.assign(prediction=check.persistence_log2))
print(f"{len(check)} conditions | forecast MAE {forecast_mae:.3f} | carry day 7 forward {carry_mae:.3f}")

42 conditions | forecast MAE 0.286 | carry day 7 forward 0.622


## 3. A batch whose controls were still silent at day 7

On 2017-10-04, every plate's untreated controls had a day-7 median of zero network spikes. The cultures had not started firing, so every control-relative day-7 value is degenerate. The batch rule flags this before any day-12 data exist.

The rule marks a degenerate reference; it does not claim the forecast is wrong. Across the 41 batches v2 has seen, it fires on 3, and on 2 of them the forecast still beat carry-forward (report §7.8).

In [5]:
silent_table, silent_summary = t2.triage(pd.read_csv(root / "examples/v2/reserve_20171004_low_quality_day5_day7.csv"), bundle)
b = silent_summary["batches"][0]
print(f"batch {b['batch']}: day-7 reference {b['day7_reference_activity_network_spikes']:.1f} network spikes")
print("batch verdict:", b["batch_verdict"])
print(silent_table.verdict.value_counts().to_string())

batch 20171004: day-7 reference 0.0 network spikes
batch verdict: day-7 reference activity is too low to normalize against; this batch's day-7 values and forecasts are unverified, so measure day 12 directly
verdict
batch reference unusable: measure day 12 directly    84


## 4. Sealed test 2: 24 earlier batches, 105 chemicals never seen

The external test was sealed before any v2 modelling, with its criteria hashed in advance, and scored once. The metric is date-macro MAE: error is averaged over doses within a chemical, then over chemicals within a batch, then equally over batches.

In [6]:
import numpy as np

keep = lambda f: f.loc[f.primary.astype(bool) & f.usable.astype(bool)].reset_index(drop=True)
b = keep(scored)
a = keep(pd.read_csv(root / "evaluation/external/arm_a_cases.csv", dtype={"identity": str}))
per_batch = pd.DataFrame({
    "v2 forecast": gate.date_losses(b),
    "carry day 7 forward": gate.date_losses(b.assign(prediction=b.day7)),
    "frozen v1 tool": gate.date_losses(a),
})
print(per_batch.mean().round(3).to_string())
wins = int((per_batch["v2 forecast"] < per_batch["carry day 7 forward"]).sum())
print(f"v2 better than carry-forward on {wins} of {len(per_batch)} batches")

# Paired batch resampling, as in the scoring code (10,000 resamples, seed 20260927).
index = np.random.default_rng(20260927).integers(len(per_batch), size=(10000, len(per_batch)))
diff = per_batch["v2 forecast"].to_numpy()[index].mean(axis=1) - per_batch["carry day 7 forward"].to_numpy()[index].mean(axis=1)
print("95% interval of v2 minus carry-forward:", np.quantile(diff, [0.025, 0.975]).round(3))

v2 forecast            0.735
carry day 7 forward    0.980
frozen v1 tool         0.941
v2 better than carry-forward on 19 of 24 batches
95% interval of v2 minus carry-forward: [-0.396 -0.123]


## 5. The trust verdict: abstention, and how it compares with a simple rule

In [7]:
error = abs(b.prediction - b.target)
carry_error = abs(b.day7 - b.target)
declined = b.declined.astype(bool)
print(f"declined {declined.sum()} of {len(b)} conditions ({declined.mean():.0%})")
print(f"v2 forecast:         kept MAE {error[~declined].mean():.3f}   all {error.mean():.3f}")
print(f"carry-forward, same: kept MAE {carry_error[~declined].mean():.3f}   all {carry_error.mean():.3f}")
extreme = np.zeros(len(b), dtype=bool)
extreme[np.argsort(-abs(b.prediction.to_numpy()), kind="stable")[: declined.sum()]] = True
print(f"declining the {declined.sum()} most extreme forecasts instead: kept MAE {error[~extreme].mean():.4f} (trust verdict {error[~declined].mean():.4f})")
print(f"every declined condition was already suppressed at day 7: highest day-7 value {b.day7[declined].max():.2f}")

declined 135 of 927 conditions (15%)
v2 forecast:         kept MAE 0.541   all 0.742
carry-forward, same: kept MAE 0.655   all 0.932
declining the 135 most extreme forecasts instead: kept MAE 0.5405 (trust verdict 0.5413)
every declined condition was already suppressed at day 7: highest day-7 value -1.30


The two ways of declining tie at the shipped threshold. Abstention itself is the robust result: it helps every predictor, including carry-forward. A declined condition was already strongly suppressed at day 7. What remains undetermined is whether it recovers or progresses, and day 12 decides that.

## 6. Post hoc: where the forecast beats carry-forward

This analysis was not pre-registered. It describes predictions that were already scored, and it selects nothing.

In [8]:
states = {
    "strongly suppressed (<= -1)": b.day7 <= -1,
    "suppressed (-1 to -0.5)": (b.day7 > -1) & (b.day7 <= -0.5),
    "near control (within 0.5)": b.day7.abs() < 0.5,
    "raised (0.5 to 1)": (b.day7 >= 0.5) & (b.day7 < 1),
    "hyperactive (>= 1)": b.day7 >= 1,
}
pd.DataFrame([{"conditions": int(m.sum()), "v2 forecast MAE": error[m].mean(), "carry-forward MAE": carry_error[m].mean(),
               "mean change day 7 to 12": (b.target - b.day7)[m].mean()} for m in states.values()], index=list(states)).round(3)

,conditions,v2 forecast MAE,carry-forward MAE,mean change day 7 to 12
strongly suppressed (<= -1),207,1.595,2.271,-0.865
suppressed (-1 to -0.5),90,0.652,0.544,0.229
near control (within 0.5),570,0.476,0.485,-0.205
raised (0.5 to 1),47,0.438,0.871,-0.865
hyperactive (>= 1),13,0.572,2.110,-2.110


Raised activity at day 7 is transient: it falls back by day 12. The forecast anticipates that fall, and carry-forward cannot. Near control, most conditions sit and the two tie.

## 7. Sealed test 1: the test the forecast lost

The earlier v1 model was scored once on three later batches from 2017, and carrying day 7 forward won. That model is retired, and the loss stays in every summary.

In [9]:
reserve = pd.read_csv(root / "evaluation/reserve_predictions.csv", dtype={"identity": str})
losses = pd.DataFrame({m: gate.date_losses(g) for m, g in reserve.groupby("model")})
names = {"extra_trees_relative_day7": "v1 forecast", "persistence_day7": "carry day 7 forward"}
view = losses[list(names)].rename(columns=names)
view.loc["mean"] = view.mean()
view.round(4)

,v1 forecast,carry day 7 forward
date,,
20170920,0.5834,0.4244
20171004,1.0573,0.6799
20171011,0.6742,0.7221
mean,0.7716,0.6088


## 8. Optional: rebuild the external test's development stage from EPA's files

This cell downloads the pinned EPA refinement files (about 3 MB, checksummed). It then re-runs `prepare` and `develop`, which takes about three minutes with four processes, and compares the results with the shipped development outputs. The comparison uses contents, not bytes, because CSV line endings differ between operating systems.

It does not re-score the external test, which was scored once and is spent.

In [10]:
RUN_FULL = False

if RUN_FULL:
    import joblib
    for step in (["tools/fetch_epa_refinement.py", "--out", "raw/epa_nfa_refinement"],
                 ["tools/reproduce_external.py", "prepare", "--raw", "raw/epa_nfa_refinement", "--out", "runs/external"],
                 ["tools/reproduce_external.py", "develop", "--raw", "raw/epa_nfa_refinement", "--out", "runs/external"]):
        subprocess.run([sys.executable, "-X", "utf8", "-B", *step], cwd=root, check=True)
    run = root / "runs/external"
    for mine, shipped in [("prepared/dev_features.csv", "evaluation/v2_prepared/dev_features.csv"),
                          ("prepared/ext_features.csv", "evaluation/v2_prepared/ext_features.csv"),
                          ("dev_oof_predictions.csv", "evaluation/v2_development/dev_oof_predictions.csv")]:
        pd.testing.assert_frame_equal(pd.read_csv(run / mine), pd.read_csv(root / shipped), check_exact=True)
        print("identical:", mine)
    features = pd.read_csv(root / "evaluation/v2_prepared/ext_features.csv")
    for name in ["final_small_refit", "final_sigma_v2"]:
        rebuilt, shipped = joblib.load(run / f"models/{name}.joblib"), joblib.load(root / f"models/v2/{name}.joblib")
        assert (rebuilt["model"].predict(features[rebuilt["columns"]]) == shipped["model"].predict(features[shipped["columns"]])).all()
        print("identical predictions:", name)

## What this does not establish

- **Organ chips, human cells, clinical toxicity.** These are rat cortical cultures in 48-well plates, not perfused organ chips, human cells or clinical toxicity.
- **Savings.** No saved recording days, wells or cost are validated.
- **Intervals.** They covered 70% of outcomes at a nominal 80% under batch shift.
- **Early warning.** The ranking for conditions still quiet at day 7 did not replicate and is withdrawn.
- **Where the gain comes from.** The v2 gain over v1 comes from refitting on more batches, not from a new model class.

The technical report (`docs/technical_report.md`) has the full evidence, the limitations and the path to organ-chip use.